In [2]:
print("Fraud Detection Project Started")

Fraud Detection Project Started


In [3]:
import pandas as pd

In [4]:
file_path = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [5]:
df.shape

(6362620, 11)

In [6]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [8]:
df.isnull().sum()


step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [9]:
df["isFraud"].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [10]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [11]:
pd.crosstab(df["type"], df["isFraud"])

isFraud,0,1
type,,
CASH_IN,1399284,0
CASH_OUT,2233384,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,528812,4097


In [12]:
df.groupby("isFraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,1.781970e+05,5.962370e+05,0.01,13368.395,74684.72,208364.76,92445516.64
1,8213.0,1.467967e+06,2.404253e+06,0.00,127091.330,441423.44,1517771.48,10000000.00


In [13]:
df["balance_error_orig"] = (
    df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
)

df["balance_error_orig"].describe()

count    6.362620e+06
mean    -2.010925e+05
std      6.066505e+05
min     -9.244552e+07
25%     -2.496411e+05
50%     -6.867726e+04
75%     -2.954230e+03
max      1.000000e-02
Name: balance_error_orig, dtype: float64

In [14]:
df.groupby("isFraud")["balance_error_orig"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,-201338.558109,606928.890826,-92445516.64,-249953.43,-69049.31,-3034.305,1.000000e-02
1,8213.0,-10692.325265,265146.131130,-10000000.00,0.00,0.00,0.000,3.725290e-09


In [15]:
df["balance_error_dest"] = (
    df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
)

df["balance_error_dest"].describe()

count    6.362620e+06
mean     5.556717e+04
std      4.415288e+05
min     -7.588573e+07
25%      0.000000e+00
50%      3.500490e+03
75%      2.935305e+04
max      1.319123e+07
Name: balance_error_dest, dtype: float64

In [16]:
df.groupby("isFraud")["balance_error_dest"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,54692.231734,4.360026e+05,-75885725.63,0.0,3500.68,29259.805,13191233.98
1,8213.0,732509.301069,1.867748e+06,-8875516.29,0.0,2231.46,442722.010,10000000.00


In [17]:
df.groupby("isFraud")["step"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,243.235663,142.140194,1.0,156.0,239.0,334.0,718.0
1,8213.0,368.413856,216.388690,1.0,181.0,367.0,558.0,743.0


In [18]:
df[df["isFraud"] == 1]["step"].value_counts().sort_index()

step
1      16
2       8
3       4
4      10
5       6
       ..
739    10
740     6
741    22
742    14
743     8
Name: count, Length: 741, dtype: int64

In [19]:
df["isFlaggedFraud"].value_counts()

isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

In [20]:
fraud_rate_by_type = (
    df.groupby("type")["isFraud"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

fraud_rate_by_type

type
TRANSFER    0.768799
CASH_OUT    0.183955
CASH_IN     0.000000
DEBIT       0.000000
PAYMENT     0.000000
Name: isFraud, dtype: float64

In [21]:
df["amount_bin"] = pd.qcut(
    df["amount"],
    q=5,
    duplicates="drop"
)

amount_fraud_rate = (
    df.groupby("amount_bin", observed=True)["isFraud"]
    .mean()
    .mul(100)
)

amount_fraud_rate

amount_bin
(-0.001, 9866.158]          0.021689
(9866.158, 36371.35]        0.040314
(36371.35, 122563.784]      0.095008
(122563.784, 246611.22]     0.085185
(246611.22, 92445516.64]    0.403214
Name: isFraud, dtype: float64

In [22]:
df.groupby("type")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
CASH_IN,1399284.0,168920.242004,1.265083e+05,0.04,70510.1825,143427.710,239899.0875,1915267.90
CASH_OUT,2237500.0,176273.964346,1.753297e+05,0.00,72669.6500,147072.185,246539.4775,10000000.00
DEBIT,41432.0,5483.665314,1.331854e+04,0.55,1500.1800,3048.990,5479.1750,569077.51
PAYMENT,2151495.0,13057.604660,1.255645e+04,0.02,4383.8200,9482.190,17561.2200,238637.98
TRANSFER,532909.0,910647.009645,1.879574e+06,2.60,215905.3500,486308.390,974958.0000,92445516.64
